In [ ]:
# Import polars for parquet processing and set Debug level do verbose
import polars as pl
pl.Config.set_verbose(True)

In [ ]:
# Scan the parquet file, set the following schema to greater reduce memory usage
active_repos_cumulative_stats_by_day = pl.scan_parquet(source='daily_1000_stars.parquet',
                                                       schema={
                                                           "repo_name": pl.String,
                                                           "day": pl.Date,
                                                           "total_stars": pl.UInt32,
                                                           "total_forks": pl.UInt32,
                                                           "total_issues_opened": pl.UInt32,
                                                           "total_issues_closed": pl.UInt32,
                                                           "total_prs_opened": pl.UInt32,
                                                           "total_prs_merged": pl.UInt32,
                                                           "total_commits": pl.UInt32,
                                                           "total_comments": pl.UInt32
                                                       })

In [20]:
metric_cols = [
    'total_stars', 'total_forks', 'total_issues_opened',
    'total_prs_opened', 'total_commits', 
]

In [ ]:
active_repos_cumulative_stats_by_day.set_sorted(['repo_name', 'day'])

In [22]:
repositories_with_rolling_metrics = (active_repos_cumulative_stats_by_day
                                     # Cast to Float (requirement for all subsequent math)
                                     .with_columns([pl.col(c).cast(pl.Float32).alias(c) for c in metric_cols])

                                     # Lags and Rolling Statistics
                                     .with_columns(
    # Lags (1, 7, 30, 60  days) 
    [pl.col(c).shift(1).over('repo_name').alias(f"{c}_lag_1d") for c in metric_cols] +
    [pl.col(c).shift(7).over('repo_name').alias(f"{c}_lag_7d") for c in metric_cols] +
    [pl.col(c).shift(30).over('repo_name').alias(f"{c}_lag_30d") for c in metric_cols] +
    [pl.col(c).shift(60).over('repo_name').alias(f"{c}_lag_60d") for c in metric_cols] +
    # Leads (30, 60, 90, 180  days) 
    [pl.col(c).shift(-30).over('repo_name').alias(f"{c}_lead_30d") for c in metric_cols] +
    [pl.col(c).shift(-60).over('repo_name').alias(f"{c}_lead_60d") for c in metric_cols] +
    [pl.col(c).shift(-90).over('repo_name').alias(f"{c}_lead_90d") for c in metric_cols] +
    [pl.col(c).shift(-180).over('repo_name').alias(f"{c}_lead_180d") for c in metric_cols] +

    # Growth Rates (Pct Change; 1, 7  days)
    [pl.col(c).pct_change(n=1).over('repo_name').alias(f"{c}_growth_1d") for c in metric_cols] +
    [pl.col(c).pct_change(n=7).over('repo_name').alias(f"{c}_growth_7d") for c in metric_cols] +

    # Rolling Means (7, 30 days)
    [pl.col(c).rolling_mean(window_size=7).over('repo_name').alias(f"{c}_rolling_mean_7d") for c in metric_cols] +
    [pl.col(c).rolling_mean(window_size=30).over('repo_name').alias(f"{c}_rolling_mean_30d") for c in metric_cols] +

    # Rolling Stds (7, 30 days)
    [pl.col(c).rolling_std(window_size=7).over('repo_name').alias(f"{c}_rolling_std_7d") for c in metric_cols] +
    [pl.col(c).rolling_std(window_size=30).over('repo_name').alias(f"{c}_rolling_std_30d") for c in metric_cols]
)

                                     # 4. Net Change (Row-wise math, NO .over needed here as columns are aligned)
                                     .with_columns([
    (pl.col(c) - pl.col(f"{c}_lag_1d")).alias(f"{c}_daily_change")
    for c in metric_cols
])

                                     # 5. Cleanup
                                     .with_columns([
    pl.when(pl.col(pl.Float32).is_infinite())
    .then(None)
    .otherwise(pl.col(pl.Float32))
    .name.keep()
])
                                     .fill_nan(0).fill_null(0)
                                     )

In [ ]:
print("Starting processing... this may take a while.")

(repositories_with_rolling_metrics
    .drop(pl.col("^.*growth.*$"))
    .filter(pl.col("total_stars_lead_180d") != 0)
    .filter(pl.col("total_stars_lag_60d") != 0)
 .sink_parquet('github_features.parquet'))

print("Done! File saved.")

In [ ]:
demo_read = pl.scan_parquet('github_features.parquet');

In [ ]:
demo_read.head(100).collect().to_pandas()